# 🗣️ MCP로 받고 SQLite에 적재하기 — Student

에이전트가 받은 값을 코드로 남기고, 삼성전자의 재무 지표까지 확인한다.

> **🖥️ 환경 설정**
>
> - **로컬**: VS Code에서 이 노트북을 열고 `FD_26/.venv` 커널을 선택한다. 실행·설치는 모두 `FD_26`에서 한다.
>
> - **의존성**: `pandas`·`requests`·`python-dotenv`·`supabase`가 없으면 `FD_26` 루트에서 `uv add pandas requests python-dotenv supabase`를 한 번 실행한다(이미 있으면 건너뜀).
>
> - **기본 런타임**: VS Code Copilot Chat(에이전트 모드)이 기본이다. Claude Code·Codex·Agy를 이미 쓰고 있다면 그걸 그대로 쓴다.
>
> - **데이터**: 이 노트북은 같은 폴더의 `data/`에 있는 `market_data.db`를 사용한다.
>
> **⌨️ 단축키 안내**
>
> | 단축키 | 동작 |
> |--------|------|
> | `Shift + Enter` | 셀 실행 후 다음 셀 이동 |
> | `Esc → A` | 위에 셀 삽입 |
> | `Esc → B` | 아래에 셀 삽입 |
> | `Esc → DD` | 셀 삭제 |

In [ ]:
# 🖥️ 환경 설정
# - VS Code에서 FD_26/.venv 커널을 선택했는지 먼저 확인한다 (실행·설치는 전부 FD_26에서).

import os
import sys

print(sys.executable)
# 예상 출력: (자기 프로젝트 경로)/FD_26/.venv/bin/python3


## 📥 자연어로 받고, 코드로 재현 가능하게 남긴다

[→ §500 강의노트](../notes/FD-04-500__load-and-measure.md)

§410에서 설치와 검증을 마쳤다면, 에이전트에게 받은 삼성전자 재무제표를 SQLite에 남긴다. 먼저 같은 종목·연도에 한 행만 남는 표를 준비한다.

In [ ]:
# 참고: §500 노트의 ‘표 준비’ 코드 순서대로 schema 확인 → PRIMARY KEY → 기존 행 보존을 구현한다.
# 참고: §500 노트의 ‘표 준비’ 코드 순서대로 schema 확인 → PRIMARY KEY → 기존 행 보존을 구현한다.
# 📌 §500-1 📥 자연어로 받고, 코드로 재현 가능하게 남긴다
# 코드를 작성해주세요 👇
# 같은 code·year가 두 번 들어가도 한 행만 남도록 표를 준비한다.

# 예상 출력:
# 예상 출력(PK 없는 옛 표였다면): financials_old에서 seed 행을 새 PK 표로 옮겼다.
# 예상 출력(이미 PK가 있으면): (출력 없이 종료)

> **학생이 보내는 요청**
>
> "삼성전자(corp_code 00126380) 2025년 사업보고서 연결(CFS) 재무제표에서 매출액, 영업이익, 당기순이익, 자본총계를 가져와줘. 단위는 억원으로 알려줘."

같은 종목·연도를 다시 받아도 행이 늘지 않게 `ON CONFLICT`로 갱신한다. 에이전트의 답을 확인한 뒤, 단위가 억원인지 확인하고 딕셔너리에 옮긴다.
> **⏱️ 20분 지점 체크포인트**
>
> MCP가 안 되면 아래 삼성전자 2025 연결 기준값(억원, §500-3 예상 출력과 동일)으로 cell 5부터 계속한다: revenue 3336059 · operating_profit 436010 · net_income 452068(700에서 직접 받은 값과 같다) · equity 4363203(700과 동일).

In [ ]:
# 참고: §500 노트의 에이전트 답을 code/year와 네 금액 열의 딕셔너리로 만든 뒤 ON CONFLICT로 저장한다.
# 참고: §500 노트의 에이전트 답을 code/year와 네 금액 열의 딕셔너리로 만든 뒤 ON CONFLICT로 저장한다.
# 📌 §500-2 📥 자연어로 받고, 코드로 재현 가능하게 남긴다
# 코드를 작성해주세요 👇
# 에이전트가 준 네 값을 억원 단위로 확인한 뒤 ON CONFLICT로 적재한다.

# 예상 출력:
# 예상 출력: (에러 없이 종료)

## 🔍 적재됐는지 SELECT로 확인

[→ §500 강의노트](../notes/FD-04-500__load-and-measure.md)

두 번 실행해도 삼성전자 2025 행 수가 정확히 1인지 확인한다. 이것이 같은 키를 다시 받아도 한 행만 남는 멱등 적재다.

In [ ]:
# 참고: §500 노트의 SELECT 예처럼 code='005930', year=2025를 조회하고 행 수가 1인지 확인한다.
# 참고: §500 노트의 SELECT 예처럼 code='005930', year=2025를 조회하고 행 수가 1인지 확인한다.
# 📌 §500-3 🔍 적재됐는지 SELECT로 확인
# 코드를 작성해주세요 👇
# SELECT 결과와 같은 종목·연도의 행 개수를 함께 출력한다.

# 예상 출력:
# 예상 출력: 삼성전자 2025 행이 정확히 1개

## 📐 시가총액을 구해 세 지표를 완성한다

[→ §500 강의노트](../notes/FD-04-500__load-and-measure.md)

적재한 재무 숫자와 최신 종가·보통주 주식수로 시가총액을 만든다. 모든 금액 단위를 억원으로 맞춘 뒤 PER·PBR·ROE를 읽는다.

In [ ]:
# 참고: §500 노트의 시가총액·PER·PBR·ROE 계산을 순서대로 적는다. 모든 금액은 억원으로 맞춘다.
# 참고: §500 노트의 시가총액·PER·PBR·ROE 계산을 순서대로 적는다. 모든 금액은 억원으로 맞춘다.
# 📌 §500-4 📐 시가총액을 구해 세 지표를 완성한다
# 코드를 작성해주세요 👇
# 최신 종가와 보통주 발행주식수로 시가총액을 억원으로 맞춘 뒤 세 지표를 계산한다.

# 예상 출력:
# 예상 출력: PER: 8.88 · PBR: 0.92 · ROE: 10.4%

## 다음 단계

PER·PBR·ROE는 매수·매도 판정이 아니라 데이터를 읽는 렌즈다. 다음 노트북에서는 같은 `financials` 행을 Supabase 클라우드에도 올리고 조회한다.